# model_api sur GPU Colab — soutenance RNCP 37827

Fait tourner **le code réel du dépôt** (`model_api/`, non modifié) sur un GPU Colab et
l'expose par un tunnel HTTPS. L'application Streamlit du poste de démonstration pointe
dessus et renvoie de vraies prédictions du modèle affiné.

## Avant de commencer

1. `Exécution` → `Modifier le type d'exécution` → **GPU** (T4 suffit ; L4 ou A100 si proposé).
2. Sur le poste, régénérer l'archive **après toute modification du code** (PowerShell,
   depuis `cim11-assistant`) :

```powershell
Compress-Archive -Path model_api,'Fine-Tuning\llama3_codage_cim11' -DestinationPath colab_payload.zip -Force
```

3. Prévoir **45 minutes** : le téléchargement du modèle de base occupe l'essentiel.

Colab Pro n'offre pas l'exécution en arrière-plan (réservée à Pro+) : **garder l'onglet
ouvert** jusqu'à la fin de la soutenance.


## 1. Vérifier le GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("\nIl faut un GPU. Sinon : Execution > Modifier le type d'execution.")


## 2. Installer les dépendances (~2 min)


In [ ]:
!pip install -q "transformers>=4.42.3" "peft>=0.19.1" "accelerate>=0.31.0" \
                "bitsandbytes>=0.43.1" "fastapi>=0.111.0" "uvicorn>=0.30.1" "pydantic>=2.7.4"
print('dependances installees')


## 3. Téléverser le code et l'adaptateur

Sélectionner `colab_payload.zip` (~13 Mo).


In [ ]:
import zipfile, pathlib, shutil
from google.colab import files

envoi = files.upload()
archive = pathlib.Path(next(iter(envoi)))
cible = pathlib.Path('/content/payload')
shutil.rmtree(cible, ignore_errors=True)

# Compress-Archive (Windows) ecrit des antislashs dans le zip. extractall() les prend
# pour des noms de fichiers et non des dossiers : on reconstruit l'arborescence.
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        chemin = info.filename.replace('\\', '/')
        if chemin.endswith('/'):
            continue
        dest = cible / chemin
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(z.read(info))

adaptateur = next(cible.rglob('adapter_config.json')).parent
paquet = next(cible.rglob('model_api/app/main.py')).parents[2]
print('adaptateur :', adaptateur)
print('code       :', paquet / 'model_api')
assert (paquet / 'model_api' / '__init__.py').exists(), 'paquet model_api incomplet'


## 4. Configuration

Le jeton Hugging Face est saisi masqué, jamais écrit dans le notebook. Il doit avoir
accès à `meta-llama/Meta-Llama-3-8B-Instruct` (modèle sous accès restreint).


In [ ]:
import os
from getpass import getpass

os.environ['HF_TOKEN'] = getpass('Jeton Hugging Face : ')
os.environ['MODEL_API_KEY'] = 'demo-key-portfolio'   # identique aux scripts de demo
os.environ['ADAPTER_PATH'] = str(adaptateur)
os.environ['BASE_MODEL'] = 'meta-llama/Meta-Llama-3-8B-Instruct'
print('configuration prete')


## 5. Démarrer model_api


In [ ]:
import subprocess, sys, time, urllib.request, json

journal = open('/content/uvicorn.log', 'w')
serveur = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'model_api.app.main:app',
     '--host', '127.0.0.1', '--port', '8001'],
    cwd=str(paquet), stdout=journal, stderr=subprocess.STDOUT, env=os.environ.copy())

for _ in range(40):
    time.sleep(1)
    try:
        with urllib.request.urlopen('http://127.0.0.1:8001/health', timeout=2) as r:
            print('model_api demarre :', json.load(r)); break
    except Exception:
        pass
else:
    print('ECHEC — journal :'); print(open('/content/uvicorn.log').read()[-2000:])


## 6. Préchauffer le modèle (10 à 20 min, une seule fois)

Le chargement est différé : sans cette étape, c'est la première requête du jury qui
attendrait le téléchargement. **À lancer bien avant d'entrer en salle.**


In [ ]:
CRH_TEST = (
    "Patiente de 32 ans hospitalisee pour une hypertrophie des vegetations adenoides "
    "responsable d'une obstruction nasale chronique et de ronflements nocturnes. "
    "Adenoidectomie realisee sous anesthesie generale, suites simples, sortie le jour meme."
)

requete = urllib.request.Request(
    'http://127.0.0.1:8001/predict',
    data=json.dumps({'texte_crh': CRH_TEST}).encode(),
    headers={'Content-Type': 'application/json',
             'X-API-Key': os.environ['MODEL_API_KEY']})

debut = time.time()
with urllib.request.urlopen(requete, timeout=1800) as r:
    reponse = json.load(r)
print(f"premiere prediction en {time.time() - debut:.0f} s (chargement inclus)\n")
print(json.dumps(reponse, indent=2, ensure_ascii=False))


## 7. Ouvrir le tunnel HTTPS

Le résolveur DNS de Colab ne sait pas résoudre `trycloudflare.com` : la vérification
depuis le notebook échouerait sans que le tunnel soit en cause. On attend donc
seulement la confirmation d'enregistrement, et c'est **depuis le poste de démonstration**
que l'URL doit être testée.


In [ ]:
import re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

log = pathlib.Path('/content/cloudflared.log')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8001', '--no-autoupdate'],
    stdout=log.open('w'), stderr=subprocess.STDOUT)

url = None
for _ in range(90):
    time.sleep(1)
    trouve = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log.read_text())
    if trouve:
        url = trouve.group(0); break

enregistre = 'Registered tunnel connection' in log.read_text()
if url and enregistre:
    print('=' * 72)
    print('  URL PUBLIQUE :', url)
    print('=' * 72)
    print('\nSur le poste de demonstration, dans cim11-assistant :')
    print(f'    demo_start_data_api.bat')
    print(f'    demo_start_app_streamlit_colab.bat {url}')
    print(f'\nVerifier dans le navigateur du poste : {url}/docs')
else:
    print('tunnel non etabli — journal :'); print(log.read_text()[-2000:])


## 8. Surveiller pendant la soutenance

Cette cellule maintient la session active et affiche les requêtes en direct. C'est
aussi une démonstration du monitorage (C11) si la question vient : les compteurs
affichés sont ceux exposés par `GET /metrics`.

Si le tunnel tombe, réexécuter la cellule 7 : l'URL change, il suffit de relancer
`demo_start_app_streamlit_colab.bat` avec la nouvelle.


In [ ]:
from datetime import datetime

while True:
    with urllib.request.urlopen('http://127.0.0.1:8001/metrics', timeout=10) as r:
        m = json.load(r)
    print(f"{datetime.now():%H:%M:%S} — {m['nb_requetes']} requetes, "
          f"{m['nb_erreurs']} erreurs, latence moyenne {m['latence_moyenne_ms']} ms, "
          f"{m['nb_anomalies_dp_manquant']} anomalies DP manquant")
    time.sleep(60)
